# 03 — Treino da CNN-B (Refinada)

Rede mais profunda com **data augmentation, BatchNorm, Dropout progressivo e GlobalAveragePooling**. Cada técnica ataca uma limitação do baseline. Meta: **≥ 88% de acurácia no teste**.

> Rodar no **Colab com GPU**. Treino ~8–10 min.

In [ ]:
# >>> EXECUTE ESTA CELULA APENAS NO GOOGLE COLAB <<<
# Clona o repositorio e instala as dependencias. Em ambiente local, pule.
# !git clone https://github.com/SEU_USUARIO/gs-cnn-eurosat.git
# %cd gs-cnn-eurosat
# !pip install -q -r requirements.txt

In [ ]:
import sys, os
# Garante que a raiz do projeto esteja no path para 'import src...'
CWD = os.getcwd()
ROOT = CWD if os.path.isdir(os.path.join(CWD, 'src')) else os.path.dirname(CWD)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)  # paths relativos (models/, reports/) consistentes
print('Project root:', ROOT)

In [ ]:
from src.data_loader import load_raw_data, make_splits, make_tf_datasets, CLASS_NAMES
from src.models import build_cnn_b
from src.train import compile_and_train, save_history
from src.evaluate import plot_training_curves

## Dados (mesma divisão do baseline — seed=42)

In [ ]:
images, labels = load_raw_data()
splits = make_splits(images, labels, seed=42)
train_ds, val_ds, test_ds = make_tf_datasets(splits, batch_size=64)

## Arquitetura

In [ ]:
model_b = build_cnn_b()
model_b.summary()

Compare a contagem de parâmetros com a CNN-A. O `GlobalAveragePooling2D` substitui o `Flatten`, reduzindo muito a densa final — a CNN-B costuma ter **menos** parâmetros que o baseline, mesmo sendo mais profunda.

## Treino
Além do `EarlyStopping` (patience=10), usamos `ReduceLROnPlateau` para reduzir o learning rate quando a `val_loss` estaciona.

In [ ]:
history_b = compile_and_train(
    model_b, train_ds, val_ds,
    epochs=50, lr=1e-3,
    checkpoint_path='models/cnn_b_best.keras',
    patience=10, use_reduce_lr=True,
)
save_history(history_b, 'reports/cnn_b_history.json')

## Curvas de aprendizado

In [ ]:
plot_training_curves(history_b.history, title='CNN-B (Refinada)',
                     save_path='reports/figures/cnn_b_curves.png')

## Avaliação no conjunto de teste

In [ ]:
test_loss, test_acc = model_b.evaluate(test_ds, verbose=0)
print(f'CNN-B — acurácia no teste: {test_acc:.4f} | loss: {test_loss:.4f}')
print('Meta de 88%:', 'ATINGIDA ✅' if test_acc >= 0.88 else 'NÃO atingida ❌')

**Se não atingir 88%:** aumente `epochs`, ajuste o `Dropout` (ex.: 0.4 na densa) ou adicione mais um bloco convolucional em `build_cnn_b`. Caso persista, documente a limitação no relatório (o enunciado permite justificativa técnica).